# onedrive_panama_inec_ingest


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "msal"], check=True)

import os
import re
import time
import requests
import msal
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.functions import lit

CLIENT_ID        = dbutils.secrets.get(scope="semis2_scope", key="AZURE_CLIENT_ID")
TENANT_ID        = dbutils.secrets.get(scope="semis2_scope", key="AZURE_TENANT_ID")
SCOPES           = ["https://graph.microsoft.com/Files.ReadWrite.All",
                    "https://graph.microsoft.com/User.Read"]
TOKEN_CACHE_FILE = Path("/Workspace/Shared/tmp/.sp_token_cache.bin")
GRAPH_BASE       = "https://graph.microsoft.com/v1.0"
ONEDRIVE_FOLDER  = "semi2-mortalidad/panama"
EXPECTED_FILES   = [f"panama_{year}_csv.csv" for year in range(2015, 2025)]


def _get_token() -> dict:
    cache = msal.SerializableTokenCache()
    if TOKEN_CACHE_FILE.exists():
        cache.deserialize(TOKEN_CACHE_FILE.read_bytes().decode())

    app = msal.PublicClientApplication(
        client_id=CLIENT_ID,
        authority="https://login.microsoftonline.com/common",
        token_cache=cache,
    )

    accounts = app.get_accounts()
    result   = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None

    if not result:
        flow = app.initiate_device_flow(scopes=SCOPES)
        print(flow["message"])
        result = app.acquire_token_by_device_flow(flow)

    TOKEN_CACHE_FILE.write_bytes(cache.serialize().encode())

    if "access_token" not in result:
        raise ValueError(f"Auth failed: {result.get('error_description', result)}")
    return result


def _headers() -> dict:
    return {"Authorization": f"Bearer {_get_token()['access_token']}"}


def test_connection() -> None:
    resp = requests.get(f"{GRAPH_BASE}/me", headers=_headers())
    resp.raise_for_status()
    me = resp.json()
    print(f"Connected as: {me.get('displayName')} ({me.get('userPrincipalName')})")


def upload_to_onedrive(local_path: str, folder: str = ONEDRIVE_FOLDER) -> None:
    name = Path(local_path).name
    url  = f"{GRAPH_BASE}/me/drive/root:/{folder}/{name}:/content"
    with open(local_path, "rb") as f:
        resp = requests.put(
            url,
            headers={**_headers(), "Content-Type": "application/octet-stream"},
            data=f,
        )
    resp.raise_for_status()
    print(f"Uploaded: {name} → OneDrive/{folder}/")


def list_files_in_folder(folder: str = ONEDRIVE_FOLDER) -> list[dict]:
    url  = f"{GRAPH_BASE}/me/drive/root:/{folder}:/children"
    resp = requests.get(url, headers=_headers())
    resp.raise_for_status()
    return [
        {"id": i["id"], "name": i["name"]}
        for i in resp.json().get("value", [])
        if "file" in i and i["name"].endswith(".csv")
    ]


def _select_expected_files(files: list[dict], expected_names: list[str]) -> list[dict]:
    by_name = {file["name"]: file for file in files}
    missing = [name for name in expected_names if name not in by_name]
    if missing:
        raise FileNotFoundError(f"Missing Panama CSVs in OneDrive folder: {missing}")
    return [by_name[name] for name in expected_names]


def download_from_onedrive(file_path: str, destination: str) -> None:
    url  = f"{GRAPH_BASE}/me/drive/root:/{file_path}:/content"
    resp = requests.get(url, headers=_headers(), stream=True)
    resp.raise_for_status()
    Path(destination).parent.mkdir(parents=True, exist_ok=True)
    with open(destination, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            f.write(chunk)
        f.flush()
        os.fsync(f.fileno())
    time.sleep(0.5)
    print(f"Downloaded: {file_path} → {destination}")


# ── Ejecución ──────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS sandbox")
test_connection()

import pandas as pd

workspace_tmp_dir = "/Workspace/Shared/tmp/panama"
os.makedirs(workspace_tmp_dir, exist_ok=True)

panama_files = _select_expected_files(list_files_in_folder(), EXPECTED_FILES)

df_total = None

for file in panama_files:
    onedrive_path = f"{ONEDRIVE_FOLDER}/{file['name']}"
    local_path    = f"{workspace_tmp_dir}/{file['name']}"

    download_from_onedrive(onedrive_path, local_path)

    # Databricks Connect: Python descarga al FS del cliente, pero spark.read con
    # scheme `file:` busca en el FS del cluster remoto (donde el archivo no está).
    # Se lee con pandas en el cliente y se envía al cluster vía createDataFrame.
    # Formato ancho INEC, separador ';', todas como string (inferSchema=False).
    pdf = pd.read_csv(local_path, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")

    df_file = (
        spark.createDataFrame(pdf)
        .withColumn("_source_file", lit(file["name"]))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    df_total = (
        df_file
        if df_total is None
        else df_total.unionByName(df_file, allowMissingColumns=True)
    )

df = df_total

# Delta rechaza ' ,;{}()=\t\n' en nombres de columna y el formato ancho INEC trae
# los grupos de edad con espacios ("Menores de 1", "5 a 9"…). Se sanean los
# encabezados (caracteres inválidos -> '_') solo para poder guardar el raw tal
# cual; el staging se encarga de normalizar las bandas a partir de estos nombres.
df = df.toDF(*[re.sub(r"[ ,;{}()=\t\n]+", "_", c) for c in df.columns])

display(df)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("sandbox.raw_panama")
